#  Exploratory Data Analysis — Credit Risk Dataset

This notebook explores the `credit_risk_dataset` before model training.
The goal is to understand the data structure, detect missing values, class imbalance, outliers, and key patterns that influence model choice.

## 1. Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('dark_background')
plt.rcParams['figure.figsize'] = (10, 5)

df = pd.read_csv('credit_risk_dataset.csv')
print('Shape:', df.shape)
df.head()

## 2. Basic Info

Check column types and get a statistical summary of all features.

In [ ]:
df.info()

In [ ]:
df.describe()

## 3. Missing Values

Two columns have missing values:
- `loan_int_rate` — ~10% missing (3,116 rows)
- `person_emp_length` — 895 missing

Both will be handled with **median imputation** in the preprocessing pipeline.

In [ ]:
missing = df.isnull().sum().sort_values(ascending=False)
missing = missing[missing > 0]

fig, ax = plt.subplots()
missing.plot(kind='bar', color='steelblue', ax=ax)
ax.set_title('Missing Values per Column')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

print(missing)

## 4. Target Variable — Class Imbalance

The dataset is **imbalanced**: 78% of clients did not default (0), only 22% defaulted (1).
This is a common challenge in credit scoring — defaulters are rare but the most important to detect.
To handle this, `class_weight='balanced'` was applied in Logistic Regression.

In [ ]:
counts = df['loan_status'].value_counts()
labels = ['No Default (0)', 'Default (1)']

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].bar(labels, counts.values, color=['steelblue', 'tomato'])
axes[0].set_title('Class Distribution — Count')
axes[0].set_ylabel('Count')

axes[1].pie(counts.values, labels=labels, autopct='%1.1f%%',
            colors=['steelblue', 'tomato'], startangle=90)
axes[1].set_title('Class Distribution — Percentage')

plt.tight_layout()
plt.show()

## 5. Numerical Features — Distributions

Histograms for all numerical columns.
**Notable outlier:** `person_age` has a maximum value of 144, which is clearly a data entry error.

In [ ]:
num_cols = ['person_age', 'person_income', 'person_emp_length',
            'loan_amnt', 'loan_int_rate', 'loan_percent_income',
            'cb_person_cred_hist_length']

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    axes[i].hist(df[col].dropna(), bins=40, color='steelblue', edgecolor='white')
    axes[i].set_title(col)
    axes[i].set_xlabel('')

axes[-1].set_visible(False)
plt.suptitle('Numerical Features — Distributions', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

print('Max person_age:', df['person_age'].max())

## 6. Categorical Features — Distributions

Bar charts for categorical columns to understand the composition of the dataset.

In [ ]:
cat_cols = ['person_home_ownership', 'loan_intent', 'loan_grade', 'cb_person_default_on_file']

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    counts = df[col].value_counts()
    axes[i].bar(counts.index, counts.values, color='steelblue')
    axes[i].set_title(col)
    axes[i].tick_params(axis='x', rotation=30)

plt.suptitle('Categorical Features — Distributions', fontsize=14)
plt.tight_layout()
plt.show()

## 7. Correlation Heatmap

Shows which numerical features are correlated with each other and with the target variable.
Strong correlation between `loan_percent_income` and `loan_amnt` is expected — larger loans take up more income.

In [ ]:
corr = df[num_cols + ['loan_status']].corr()

plt.figure(figsize=(10, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            square=True, linewidths=0.5)
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.show()

## 8. Default Rate by Loan Grade

`loan_grade` is an important feature — it directly reflects credit risk assessment.
We expect higher default rates in grades D, E, F, G.

In [ ]:
default_by_grade = df.groupby('loan_grade')['loan_status'].mean().sort_index()

plt.figure(figsize=(8, 4))
default_by_grade.plot(kind='bar', color='tomato', edgecolor='white')
plt.title('Default Rate by Loan Grade')
plt.ylabel('Default Rate')
plt.xlabel('Loan Grade')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 9. Key Findings

**Summary of observations from EDA:**

1. **Class imbalance** — 78% no default vs 22% default. Addressed with `class_weight='balanced'` in Logistic Regression.
2. **Missing values** — `loan_int_rate` (~10%) and `person_emp_length` handled via median imputation.
3. **Outlier in age** — `person_age` max = 144, likely a data entry error. Not removed to keep the pipeline simple, but worth noting.
4. **Loan grade is a strong predictor** — default rate clearly increases from grade A to G, confirming it is an important feature.
5. **Non-linear patterns** — the data shows non-linear relationships between features and the target, which motivated the choice of Gradient Boosting alongside Logistic Regression.